<a href="https://colab.research.google.com/github/sbleeedu1-ai/python_CLI_board_example_26_08/blob/main/%ED%8C%8C%EC%9D%B4%EC%8D%AC_CLI_board.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# 게시판
from datetime import datetime
from pytz import timezone
from dataclasses import dataclass
import getpass    # 비밀번호 보이지 않게

@dataclass
class Article:
  id: int
  regDate: str
  updateDate: str
  title: str
  body: str

@dataclass
class Member:
  id: int
  regDate: str
  updateDate: str
  loginId : str
  loginPw: str
  name: str


def now():
  return datetime.now(timezone('Asia/Seoul')).strftime("%Y-%m-%d %H:%M:%S")

def is_cmd_right(cmd):
  cmd_bits = cmd.split(" ")

  if len(cmd_bits) < 3:
    print("명령어를 다시 입력해주세요 (글 번호 없음)")
    return None

  if not cmd_bits[2].isdigit():
    print("명령어를 다시 입력해주세요 (번호대신 문자입력)")
    return None

  return int(cmd_bits[2])

def is_article_exist(articles,article_id):
  for article in articles:
    if article.id == article_id:
      return article
  return None

def format_date(regDate):
  today = now().split(" ")[0]
  write_date = regDate.split(" ")[0]
  write_time = regDate.split(" ")[1]
  if today == write_date:
    return write_time
  else:
    return write_date

def make_article_TestData():
  return [
      Article(1, "2025-12-12 12:12:12", "2025-12-12 12:12:12", "제목1", "내용1"),
      Article(2, now(), now(), "제목2", "내용2"),
      Article(3, now(), now(), "제목3", "내용3")
          ]

def make_member_TestData():
  return [
      Member(1, now(), now(), "test1", "test1", "회원1"),
      Member(2, now(), now(), "test2", "test2", "회원2")
          ]

def is_member_exist(members,member_id):
  for member in members:
    if member.loginId == member_id:
      return True
  return False

# is_article_exist 구조를 그대로 사용

# def pw_check(password):
#   for char in password:
#     if char.isupper():
#       if password.isalnum():
#         return True
#       else:
#         print("유효하지 않은 비밀번호 입니다.(특수문자 제외)")
#         return False

#   print("유효하지 않은 비밀번호 입니다.(대문자 포함 필수)")
#   return False

# isupper 를 잘못 알았다. 모든 글자가 대문자일때만 True
# 한글자씩 따와서 확인

def is_pw_right(members,mem_id,mem_pw):
  for member in members:
    if member.loginId == mem_id:
      if member.password == mem_pw:
        return True
  return False

# is_member_exist 함수와 동일하게 리스트에서 꺼내 하나씩 대조
# 맞는 아이디인 경우에만 암호 일치 체크


print("== CLI 게시판 실행 ==")
article_num = 3
articles = make_article_TestData()

member_num = 2
members = make_member_TestData()

logInStatus=False      # 로그인 상태 저장

while True:
  user_cmd = input("명령어 ) ").strip()

  if user_cmd == 'exit':
    break

  # 회원 Member
  # 회원 가입
  # member join

  elif  user_cmd == 'member join':
    member_num += 1

    while True:
      loginId = input("아이디 : ")
      memberId = is_member_exist(members, loginId)
      if memberId :
        print("이미 존재하는 아이디 입니다.")
        continue
      # 아이디 중복체크
      # -> 없으면 다시 명령어
      # 중복 체크를 통과한 경우에만 -> 비밀번호 입력
      break

    while True:
      loginPw = getpass.getpass("암호 : ")
      loginPwConfirm = getpass.getpass("암호 확인 : ")
      if loginPw != loginPwConfirm:
        print("암호가 일치하지 않습니다.")
        continue
      # 비밀번호는 가려져 보이게 getpass 사용
      # 원하는 비밀번호를 입력했는지 확인 -> 이 필요했다 (오타 방지)
      # memberPw = pw_check(loginPw)
      # if not memberPw:
      #   continue
      # # 비밀번호 유효성 체크
      # # 대문자 1개 포함, 문자숫자 자유, 특수문자 제한
      break

    name = input("이름 : ")
    member = Member(member_num,now(),now(),loginId,loginPw,name)
    members.append(member)
    print(f"{member_num}번째 회원 가입을 축하합니다.")

  # 로그인
  # member login

  elif  user_cmd == 'member login':
    if logInStatus:
      print("이미 로그인한 상태입니다.")
      continue
    # 로그인 상태를 확인
    # 한 상태이면 True 일테니 명령어 입력으로 돌려보냄

    while True:
      login_id = input("아이디 : ")
      memberId = is_member_exist(members, login_id)
      if not memberId:
        print("존재하지 않는 아이디 입니다.")
        continue
      # 아이디 존재여부 체크
      break

    while True:
      login_pw = getpass.getpass("암호 : ")
      loginPw = is_pw_right(members, login_id, login_pw)
      if not loginPw:
        print("비밀번호가 틀렸습니다.")
        continue
      # 암호는 맞는 아이디의 암호만을 체크해야함
      break

    logInStatus = True
    print("로그인 성공")
    # 다 통과 시 로그인 상태에 True 저장
    # 로그인은 회원에게만 기능을 사용할 수 있는 권한을 주는 것
    # 로그인 상태인지 아닌지만 체크하면 되니까?

  # 로그아웃
  # member logout

  elif user_cmd == 'member logout':
    if not logInStatus:
      print("로그인 하지 않은 상태입니다.")
      continue
    # 로그인 상태 확인
    # 아니면 기본 False 상태여서 not 을 붙여 if 문 실행됨

    logInStatus = False
    print("로그아웃 되었습니다.")
    # 로그인 상태면 아닌 상태로 False 처리

  elif user_cmd == 'article write':
    if not logInStatus:
      print("로그인이 필요합니다.")
      continue

    article_num += 1
    title = input("제목 : ")
    content = input("내용 : ")
    article = Article(article_num,now(),now(),title,content)
    articles.append(article)
    print(f"{article_num}번 글이 생성되었습니다.")

  elif user_cmd == 'article list':
    if not logInStatus:
      print("로그인이 필요합니다.")
      continue

    if not articles:
      print("작성한 글이 없습니다.")
    else:
      print("========================================================")
      print("번호".ljust(5),end='/')
      print("    제목".ljust(10),end='/')
      print("    내용".ljust(10),end='/')
      print("    작성시간".ljust(10))
      for a in reversed(articles):
          print(f"  {a.id}".ljust(8),end='/')
          print(f"     {a.title}".ljust(10),end='/')
          print(f"     {a.content}".ljust(10),end='/')
          print(f"     {format_date(a.regDate)}")
      print("========================================================")

  elif user_cmd.startswith('article delete'):
    if not logInStatus:
      print("로그인이 필요합니다.")
      continue

    deletedId = is_cmd_right(user_cmd)
    if deletedId is None:
      continue

    article = is_article_exist(articles, deletedId)
    if article is None:
      print(f"{deletedId}번 글은 존재하지 않습니다.")
      continue

    articles.remove(article)
    print(f"{deletedId}번 글이 삭제 되었습니다.")

  elif user_cmd.startswith('article edit'):
    if not logInStatus:
      print("로그인이 필요합니다.")
      continue

    editedId = is_cmd_right(user_cmd)
    if editedId is None:
      continue

    article = is_article_exist(articles, editedId)
    if article is None:
      print(f"{editedId}번 글은 존재하지 않습니다.")
      continue

    print(f"기존 제목 : {article.title}")
    print(f"기존 내용 : {article.content}")
    article.title = input("새 제목 : ")
    article.content = input("새 내용 : ")
    article.updateDate = now()
    print(f"{editedId}번 글이 수정 되었습니다.")

  elif user_cmd.startswith('article detail'):
    if not logInStatus:
      print("로그인이 필요합니다.")
      continue

    detailId = is_cmd_right(user_cmd)
    if detailId is None:
      continue

    article = is_article_exist(articles, detailId)
    if article is None:
      print(f"{detailId}번 글은 존재하지 않습니다.")
      continue

    print(f"번호 : {article.id}")
    print(f"작성 날짜 : {article.regDate}")
    print(f"수정 날짜 : {article.updateDate}")
    print(f"제목 : {article.title}")
    print(f"내용 : {article.content}")

  else:
    print("지원하지 않은 명령어 입니다.")

print("== CLI 게시판 종료 ==")


== CLI 게시판 실행 ==
명령어 ) article write
로그인이 필요합니다.
명령어 ) article list
로그인이 필요합니다.
명령어 ) article ecit
지원하지 않은 명령어 입니다.
명령어 ) article edit
로그인이 필요합니다.
명령어 ) article delete
로그인이 필요합니다.
명령어 ) exit
== CLI 게시판 종료 ==
